# 03 — Calibrated sector-agnostic baseline and incident episodes

**Outcome:** create a manageable ranked list of persistent incident episodes,
not thousands of independent time buckets.

This is a transparent modelling workbench. It contains a real statistical
baseline—history-only expected value and variability—plus truth-free
calibration, episode formation, ranking, and offline evaluation.

The critical boundary is:

1. read the label-free canonical profile and `SPEC-CORE`;
2. verify that both describe the same immutable canonical run;
3. fit/calibrate using each entity's early chronological window;
4. score the later window and save all outputs;
5. only then read `SPEC-EVAL`.

**Sector-agnostic logic:** dispatch by `measurement_kind` and
`expected_behaviour_profile`, apply `anomaly_direction`, use canonical
relations for grouping, and emit the same outputs in every sector. Sector
packs provide metadata; native field names never appear in the model.

## 1. Setup and research controls

Notebook 01 or 02 must have produced the selected `SPEC-CORE` first. The
default uses 20 telecom entities so iteration remains safe in Colab.

Important controls:

- `CALIBRATION_FRACTION`: early data used to estimate per-metric thresholds;
- `TARGET_POINT_QUANTILE`: expected upper tail under calibration data;
- `MIN_EPISODE_BUCKETS`: persistence needed for otherwise weak evidence;
- `HIGH_CONFIDENCE_RATIO`: score/threshold ratio that can stand alone;
- `MAX_INCIDENTS_PER_DAY`: explicit research workload budget.

These are research assumptions, not operator-approved values.

In [ ]:
import json
import math
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "week1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from week1_core import CORE_VERSION, sha256_file, write_json

SECTOR = os.getenv("MODEL_SECTOR", "telecom")
default_run = (
    DRIVE_ROOT / "outputs" / "research" / f"v{CORE_VERSION}"
    / ("telecom" if SECTOR == "telecom" else "petrobras_3w")
    / ("telecom_v4_1_full_v1" if SECTOR == "telecom" else "contract_challenge_v1")
)
CORE_RUN_ROOT = Path(os.getenv("MODEL_CORE_RUN_ROOT", str(default_run)))
CORE = CORE_RUN_ROOT / "SPEC-CORE"
EVALUATION = CORE_RUN_ROOT / "SPEC-EVAL"
PROFILE_ROOT = Path(os.getenv(
    "MODEL_PROFILE_ROOT",
    str(
        DRIVE_ROOT / "outputs" / "research" / f"v{CORE_VERSION}"
        / "profiles" / SECTOR / CORE_RUN_ROOT.name
    ),
))
MODEL_RUN_ID = os.getenv(
    "MODEL_RUN_ID", f"{SECTOR}_episode_baseline_v3"
)
MODEL_OUTPUT = (
    DRIVE_ROOT / "outputs" / "research" / f"v{CORE_VERSION}"
    / "models" / SECTOR / MODEL_RUN_ID
)

ENTITY_LIMIT = int(os.getenv("MODEL_ENTITY_LIMIT", "20"))
HISTORY = int(os.getenv("MODEL_HISTORY", "96"))
MIN_HISTORY = int(os.getenv("MODEL_MIN_HISTORY", "24"))
CALIBRATION_FRACTION = float(os.getenv("MODEL_CALIBRATION_FRACTION", "0.40"))
TARGET_POINT_QUANTILE = float(os.getenv("MODEL_POINT_QUANTILE", "0.995"))
MIN_RAW_THRESHOLD = float(os.getenv(
    "MODEL_MIN_RAW_THRESHOLD",
    os.getenv("MODEL_SCORE_THRESHOLD", "4.0"),
))
SCORE_CAP = float(os.getenv("MODEL_SCORE_CAP", "100.0"))
MIN_EPISODE_BUCKETS = int(os.getenv("MODEL_MIN_EPISODE_BUCKETS", "2"))
HIGH_CONFIDENCE_RATIO = float(os.getenv(
    "MODEL_HIGH_CONFIDENCE_RATIO", "1.5"
))
MAX_INCIDENTS_PER_DAY = int(os.getenv(
    "MODEL_MAX_INCIDENTS_PER_DAY", "10"
))

assert 0 < CALIBRATION_FRACTION < 1
assert 0 < TARGET_POINT_QUANTILE < 1
assert MIN_EPISODE_BUCKETS >= 1
assert MAX_INCIDENTS_PER_DAY >= 0

if MODEL_OUTPUT.exists():
    raise FileExistsError(
        f"Use a new MODEL_RUN_ID; refusing to overwrite {MODEL_OUTPUT}"
    )
MODEL_OUTPUT.mkdir(parents=True)

display(pd.Series({
    "sector": SECTOR,
    "spec_core": str(CORE),
    "spec_eval": str(EVALUATION),
    "canonical_profile": str(PROFILE_ROOT),
    "model_output": str(MODEL_OUTPUT),
    "entity_limit": ENTITY_LIMIT or "all",
    "history": HISTORY,
    "minimum_history": MIN_HISTORY,
    "calibration_fraction": CALIBRATION_FRACTION,
    "target_point_quantile": TARGET_POINT_QUANTILE,
    "minimum_raw_threshold": MIN_RAW_THRESHOLD,
    "minimum_episode_buckets": MIN_EPISODE_BUCKETS,
    "high_confidence_ratio": HIGH_CONFIDENCE_RATIO,
    "maximum_incidents_per_day": MAX_INCIDENTS_PER_DAY or "disabled",
}, name="value").to_frame())

## 2. Verify the canonical profile, then load only `SPEC-CORE`

The profile was created from the early, label-free part of this same canonical run. It supplies descriptive evidence and kind-based transform recommendations—not fitted anomaly thresholds. `SPEC-EVAL` remains unopened. `MODEL_ENTITY_LIMIT=0`
means all entities, but the current research notebook still combines the
selected slice in memory. Increase the limit gradually; full-population
streaming comes after the incident definition is stable.

In [ ]:
core_manifest = json.loads((CORE / "manifest.json").read_text())
profile_manifest = json.loads(
    (PROFILE_ROOT / "profile_manifest.json").read_text()
)
metric_profile = pd.read_parquet(
    PROFILE_ROOT / "metric_profile.parquet"
)
assert profile_manifest["spec_eval_opened"] is False
assert profile_manifest["contract_version"] == CORE_VERSION
assert profile_manifest["sector"] == SECTOR
assert profile_manifest["spec_core_manifest_sha256"] == sha256_file(
    CORE / "manifest.json"
)
assert profile_manifest["metric_catalogue_sha256"] == sha256_file(
    CORE / "metric_catalogue.parquet"
)
assert math.isclose(
    float(profile_manifest["profile_fraction"]),
    CALIBRATION_FRACTION,
)
catalogue = pd.read_parquet(CORE / "metric_catalogue.parquet")
registry = pd.read_parquet(CORE / "entity_registry.parquet")
relations = pd.read_parquet(CORE / "entity_relations.parquet")
gaps = pd.read_parquet(CORE / "collection_gaps.parquet")

leaf_type = "ont" if SECTOR == "telecom" else "oil_well"
leaf_entities = sorted(
    registry.loc[registry["entity_type"].eq(leaf_type), "entity_id"]
    .astype(str).unique()
)
selected_entities = (
    leaf_entities if ENTITY_LIMIT == 0
    else leaf_entities[:ENTITY_LIMIT]
)
if not selected_entities:
    raise ValueError(f"no {leaf_type!r} entities in the core run")

telemetry_parts = sorted((CORE / "telemetry").glob("part-*.parquet"))
frames = []
for part in telemetry_parts:
    frame = pd.read_parquet(part)
    frame = frame.loc[
        frame["entity_id"].astype(str).isin(selected_entities)
    ]
    if len(frame):
        frames.append(frame)
telemetry = pd.concat(frames, ignore_index=True)
telemetry["event_ts"] = pd.to_datetime(telemetry["event_ts"], utc=True)
telemetry["value"] = pd.to_numeric(telemetry["value"], errors="coerce")
telemetry["exposure"] = pd.to_numeric(
    telemetry["exposure"], errors="coerce"
)

metadata_columns = [
    "metric_id", "measurement_kind", "anomaly_direction",
    "expected_behaviour_profile",
]
telemetry = telemetry.merge(
    catalogue[metadata_columns],
    on="metric_id",
    how="left",
    validate="many_to_one",
)
assert telemetry["measurement_kind"].notna().all()
assert telemetry["anomaly_direction"].notna().all()
assert telemetry["expected_behaviour_profile"].notna().all()
recommended = metric_profile.set_index("metric_id")["recommended_transform"]
missing_profiles = set(telemetry["metric_id"]) - set(recommended.index)
assert not missing_profiles, missing_profiles

print(
    f"Loaded {len(telemetry):,} rows, {telemetry.entity_id.nunique()} "
    f"entities, {telemetry.metric_id.nunique()} metrics from SPEC-CORE only."
)

## 3. Behaviour-aware measurement transformation

The transformation follows the kind-based recommendation recorded by Notebook 00. The model verifies the recommendation rather than silently inventing a second policy:

- gauges and discrete states retain their value;
- interval counts become exposure-adjusted log rates when possible;
- cumulative counters become non-negative, reset-safe increments;
- zero-inflated bounded measurements use a log scale that separates zero
  from small positive values.

Exposure formulas remain in the sector adapter; they are not duplicated here.

In [ ]:
model_frame = telemetry.sort_values(
    ["entity_id", "metric_id", "event_ts"], kind="stable"
).copy()
model_frame["model_value"] = model_frame["value"]
model_frame["model_transform"] = "identity"

interval_count = model_frame["measurement_kind"].eq("interval_count")
has_exposure = interval_count & model_frame["exposure"].gt(0)
model_frame.loc[interval_count, "model_value"] = np.log1p(
    model_frame.loc[interval_count, "value"].clip(lower=0)
)
model_frame.loc[interval_count, "model_transform"] = "log1p_count"
model_frame.loc[has_exposure, "model_value"] = np.log(
    (model_frame.loc[has_exposure, "value"].clip(lower=0) + 0.5)
    / model_frame.loc[has_exposure, "exposure"]
)
model_frame.loc[has_exposure, "model_transform"] = "log_exposure_rate"

cumulative = model_frame["measurement_kind"].eq("cumulative_counter")
counter_diff = (
    model_frame.loc[cumulative]
    .groupby(["entity_id", "metric_id"], sort=False)["value"]
    .diff()
)
counter_diff = counter_diff.where(counter_diff.ge(0))
model_frame.loc[cumulative, "model_value"] = np.log1p(counter_diff)
model_frame.loc[cumulative, "model_transform"] = "log1p_increment"

zero_inflated = model_frame["expected_behaviour_profile"].eq(
    "zero_inflated_bounded"
)
nonnegative = model_frame.loc[zero_inflated, "value"].clip(lower=0)
model_frame.loc[zero_inflated, "model_value"] = np.log10(
    nonnegative + 1e-15
)
model_frame.loc[zero_inflated, "model_transform"] = "log10_floor"

model_frame.loc[
    model_frame["quality_code"].eq("invalid"), "model_value"
] = np.nan
model_frame["recommended_transform"] = model_frame[
    "metric_id"
].map(recommended)
transform_mismatches = model_frame.loc[
    model_frame["model_transform"].ne(
        model_frame["recommended_transform"]
    ),
    ["metric_id", "model_transform", "recommended_transform"],
].drop_duplicates()
assert transform_mismatches.empty, transform_mismatches

display(
    model_frame.groupby(
        ["expected_behaviour_profile", "model_transform"]
    )["model_value"].agg(["count", "min", "median", "max"])
)

## 4. History-only statistical model and truth-free calibration

For each entity-metric series, the baseline uses only previous observations.
Both the rolling scale and its fallback are history-only; no future values are
used.

Raw scores are then calibrated per metric using each entity's early chronological window. A calibrated score of `1.0` means the metric crossed its own
calibration threshold. This prevents naturally volatile BER/CRC channels from
dominating merely because their raw score scale differs.

In [ ]:
def score_one_series(group):
    group = group.sort_values("event_ts").copy()
    values = group["model_value"]
    shifted = values.shift(1)
    history = shifted.rolling(HISTORY, min_periods=MIN_HISTORY)

    group["baseline"] = history.median()
    q25 = history.quantile(0.25)
    q75 = history.quantile(0.75)
    iqr_scale = (q75 - q25) / 1.349

    historical_change = (
        values.diff().abs().where(lambda item: item.gt(0))
        .shift(1)
        .rolling(HISTORY, min_periods=MIN_HISTORY)
        .median()
    )
    relative_floor = (
        shifted.abs()
        .rolling(HISTORY, min_periods=MIN_HISTORY)
        .median()
        .mul(0.01)
        .clip(lower=1e-6)
    )
    fallback = pd.concat(
        [historical_change, relative_floor], axis=1
    ).max(axis=1, skipna=True)
    group["scale"] = iqr_scale.where(
        iqr_scale.ge(relative_floor), fallback
    )
    group["signed_score"] = (
        (group["model_value"] - group["baseline"]) / group["scale"]
    )

    direction = group["anomaly_direction"].iloc[0]
    signed = group["signed_score"]
    if direction == "increase":
        group["raw_anomaly_score"] = signed.clip(lower=0)
    elif direction == "decrease":
        group["raw_anomaly_score"] = (-signed).clip(lower=0)
    else:
        group["raw_anomaly_score"] = signed.abs()
    group["raw_anomaly_score"] = group[
        "raw_anomaly_score"
    ].clip(upper=SCORE_CAP)
    return group

scored = pd.concat(
    [
        score_one_series(group)
        for _, group in model_frame.groupby(
            ["entity_id", "metric_id"], sort=False
        )
    ],
    ignore_index=True,
)

### 4.1 Per-entity calibration windows

This is the fit boundary. Every entity contributes its early window to metric-level threshold calibration; later rows are reserved for prospective scoring.

In [ ]:
scored["phase"] = "evaluation"
scored["calibration_end"] = pd.Series(
    pd.NaT, index=scored.index, dtype="datetime64[ns, UTC]"
)
calibration_window_rows = []
for entity_id, entity_rows in scored.groupby("entity_id", sort=True):
    entity_times = (
        entity_rows["event_ts"].dropna().drop_duplicates().sort_values()
    )
    if len(entity_times) < 3:
        raise ValueError(
            f"{entity_id} has fewer than three unique timestamps"
        )
    calibration_index = max(
        0,
        min(
            len(entity_times) - 2,
            int(len(entity_times) * CALIBRATION_FRACTION) - 1,
        ),
    )
    entity_calibration_end = pd.Timestamp(
        entity_times.iloc[calibration_index]
    )
    scored.loc[entity_rows.index, "calibration_end"] = (
        entity_calibration_end
    )
    calibration_indexes = entity_rows.index[
        entity_rows["event_ts"].le(entity_calibration_end)
    ]
    scored.loc[calibration_indexes, "phase"] = "calibration"
    calibration_window_rows.append({
        "entity_id": str(entity_id),
        "observation_start": entity_times.iloc[0],
        "observation_end": entity_times.iloc[-1],
        "unique_timestamps": int(len(entity_times)),
        "calibration_end": entity_calibration_end,
    })

calibration_windows = pd.DataFrame(calibration_window_rows)
calibration_windows.to_parquet(
    MODEL_OUTPUT / "calibration_windows.parquet", index=False
)
calibration_end_min = calibration_windows["calibration_end"].min()
calibration_end_max = calibration_windows["calibration_end"].max()

threshold_rows = []
for metric_id, group in scored.groupby("metric_id", sort=True):
    calibration_scores = group.loc[
        group["phase"].eq("calibration"),
        "raw_anomaly_score",
    ].dropna()
    empirical = calibration_scores.quantile(TARGET_POINT_QUANTILE)
    if not math.isfinite(empirical):
        empirical = MIN_RAW_THRESHOLD
    threshold_rows.append({
        "metric_id": metric_id,
        "calibration_rows": int(len(calibration_scores)),
        "empirical_quantile": float(empirical),
        "raw_score_threshold": float(max(
            empirical, MIN_RAW_THRESHOLD
        )),
    })
calibration_thresholds = pd.DataFrame(threshold_rows)

scored = scored.merge(
    calibration_thresholds[["metric_id", "raw_score_threshold"]],
    on="metric_id",
    how="left",
    validate="many_to_one",
)
scored["calibrated_score"] = (
    scored["raw_anomaly_score"] / scored["raw_score_threshold"]
)
scored["is_point_alert"] = (
    scored["phase"].eq("evaluation")
    & scored["calibrated_score"].ge(1.0)
)

### 4.2 Freeze scores and metric diagnostics

These artifacts are written before any evaluation table is opened. `model_diagnostics` shows whether one metric is creating a disproportionate share of alerts.

In [ ]:
score_columns = [
    "event_ts", "entity_id", "metric_id", "value", "quality_code",
    "expected_behaviour_profile", "model_transform", "model_value",
    "baseline", "scale", "signed_score", "raw_anomaly_score",
    "raw_score_threshold", "calibrated_score", "phase",
    "calibration_end", "is_point_alert",
]
scores_to_save = scored[score_columns].copy()
scores_to_save.to_parquet(
    MODEL_OUTPUT / "anomaly_scores.parquet", index=False
)
calibration_thresholds.to_parquet(
    MODEL_OUTPUT / "calibration_thresholds.parquet", index=False
)

diagnostic_rows = []
for metric_id, group in scored.groupby("metric_id", sort=True):
    evaluation_group = group.loc[group["phase"].eq("evaluation")]
    diagnostic_rows.append({
        "metric_id": metric_id,
        "expected_behaviour_profile": group[
            "expected_behaviour_profile"
        ].iloc[0],
        "model_transform": group["model_transform"].iloc[0],
        "rows": int(len(group)),
        "evaluation_rows": int(len(evaluation_group)),
        "threshold": float(group["raw_score_threshold"].iloc[0]),
        "point_alert_rows": int(
            evaluation_group["is_point_alert"].sum()
        ),
        "point_alert_rate": float(
            evaluation_group["is_point_alert"].mean()
        ) if len(evaluation_group) else None,
    })
model_diagnostics = pd.DataFrame(diagnostic_rows)
model_diagnostics.to_parquet(
    MODEL_OUTPUT / "model_diagnostics.parquet", index=False
)

display(calibration_thresholds)
display(model_diagnostics.sort_values(
    "point_alert_rate", ascending=False
))
print(
    "Entity calibration end range:",
    calibration_end_min,
    "to",
    calibration_end_max,
)
print("Saved scores before evaluation truth was read.")

## 5. Form persistent incident episodes

Point alerts first become domain/time buckets. Adjacent buckets for the same
domain are then merged into one episode. An episode is retained when it has at
least one form of meaningful evidence:

- persistence across multiple buckets;
- more than one affected entity;
- more than one anomalous metric;
- a high calibrated score.

Episode duration uses the native measurement cadence; the larger grouping bucket is reported separately and never presented as observed duration.

Ranking uses calibrated severity, persistence, corroboration and priority.
A daily budget is applied only after all candidate episodes are saved, so its
effect remains auditable.

In [ ]:
network_relations = relations.loc[
    relations.get(
        "relation_family",
        pd.Series(index=relations.index, dtype="object"),
    ).eq("network_topology")
]
child_to_parent = (
    network_relations.drop_duplicates("child_entity_id")
    .set_index("child_entity_id")["parent_entity_id"]
    .astype(str).to_dict()
    if len(network_relations) else {}
)

priorities = {}
for row in registry.itertuples(index=False):
    attributes = json.loads(row.attributes_json or "{}")
    priorities[str(row.entity_id)] = (
        float(attributes.get("service_impact_weight", 1.0) or 1.0)
        * float(attributes.get("customer_priority_weight", 1.0) or 1.0)
    )

flagged = scored.loc[scored["is_point_alert"]].copy()
flagged["domain_entity_id"] = flagged["entity_id"].map(
    lambda entity: child_to_parent.get(str(entity), str(entity))
)
flagged["priority_weight"] = (
    flagged["entity_id"].map(priorities).fillna(1.0)
)
flagged["is_measured"] = flagged["quality_code"].eq("measured")

cadence_seconds = float(core_manifest.get(
    "cadence_seconds",
    1 if SECTOR == "petrobras_3w" else 900,
))
if cadence_seconds >= 60:
    bucket_frequency = "h"
    bucket_delta = pd.Timedelta(hours=1)
else:
    bucket_frequency = "min"
    bucket_delta = pd.Timedelta(minutes=1)
flagged["bucket_start"] = flagged["event_ts"].dt.floor(bucket_frequency)

bucket_rows = []
for (domain, bucket), group in flagged.groupby(
    ["domain_entity_id", "bucket_start"], sort=True
):
    affected = sorted(group["entity_id"].astype(str).unique())
    metrics = sorted(group["metric_id"].astype(str).unique())
    bucket_rows.append({
        "domain_entity_id": str(domain),
        "bucket_start": bucket,
        "first_event_ts": group["event_ts"].min(),
        "last_event_ts": group["event_ts"].max(),
        "point_count": int(len(group)),
        "affected_entities": "|".join(affected),
        "anomalous_metrics": "|".join(metrics),
        "max_score": float(group["raw_anomaly_score"].max()),
        "mean_score": float(group["raw_anomaly_score"].mean()),
        "max_calibrated_score": float(
            group["calibrated_score"].max()
        ),
        "mean_calibrated_score": float(
            group["calibrated_score"].mean()
        ),
        "max_priority_weight": float(group["priority_weight"].max()),
        "measured_points": int(group["is_measured"].sum()),
    })
bucket_alerts = pd.DataFrame(bucket_rows)

### 5.1 Merge adjacent buckets into episodes

This cell defines one episode record and merges consecutive buckets within the same operational domain.

In [ ]:
def finish_episode(domain, rows):
    entities = sorted(set().union(*[
        set(row["affected_entities"].split("|")) for row in rows
    ]))
    metrics = sorted(set().union(*[
        set(row["anomalous_metrics"].split("|")) for row in rows
    ]))
    point_count = sum(row["point_count"] for row in rows)
    bucket_count = len(rows)
    max_calibrated = max(row["max_calibrated_score"] for row in rows)
    reasons = []
    if bucket_count >= MIN_EPISODE_BUCKETS:
        reasons.append("persistent")
    if len(entities) > 1:
        reasons.append("multi_entity")
    if len(metrics) > 1:
        reasons.append("multi_metric")
    if max_calibrated >= HIGH_CONFIDENCE_RATIO:
        reasons.append("high_confidence")

    episode_start = min(row["first_event_ts"] for row in rows)
    last_observation = max(row["last_event_ts"] for row in rows)
    episode_end = last_observation + pd.Timedelta(
        seconds=cadence_seconds
    )
    duration_minutes = (
        episode_end - episode_start
    ).total_seconds() / 60
    bucket_span_minutes = (
        rows[-1]["bucket_start"] + bucket_delta
        - rows[0]["bucket_start"]
    ).total_seconds() / 60
    measured_points = sum(row["measured_points"] for row in rows)
    mean_raw = np.average(
        [row["mean_score"] for row in rows],
        weights=[row["point_count"] for row in rows],
    )
    mean_calibrated = np.average(
        [row["mean_calibrated_score"] for row in rows],
        weights=[row["point_count"] for row in rows],
    )
    max_priority = max(row["max_priority_weight"] for row in rows)
    measured_fraction = measured_points / max(point_count, 1)

    rank_score = (
        2.0 * math.log1p(max_calibrated)
        + 0.8 * math.log1p(bucket_count)
        + 1.2 * math.log1p(len(entities))
        + 0.8 * math.log1p(len(metrics))
        + 0.1 * math.log1p(max_priority)
        + 0.25 * measured_fraction
    )
    return {
        "incident_start": episode_start,
        "incident_end": episode_end,
        "duration_minutes": duration_minutes,
        "bucket_span_minutes": bucket_span_minutes,
        "domain_entity_id": domain,
        "bucket_count": bucket_count,
        "point_count": point_count,
        "affected_entity_count": len(entities),
        "affected_entities": "|".join(entities),
        "anomalous_metric_count": len(metrics),
        "anomalous_metrics": "|".join(metrics),
        "max_score": max(row["max_score"] for row in rows),
        "mean_score": float(mean_raw),
        "max_calibrated_score": max_calibrated,
        "mean_calibrated_score": float(mean_calibrated),
        "max_priority_weight": max_priority,
        "measured_fraction": measured_fraction,
        "eligibility_reason": "|".join(reasons),
        "eligible": bool(reasons),
        "rank_score": rank_score,
    }

episode_rows = []
if len(bucket_alerts):
    for domain, group in bucket_alerts.groupby(
        "domain_entity_id", sort=True
    ):
        current = []
        previous_bucket = None
        for record in group.sort_values("bucket_start").to_dict("records"):
            if (
                previous_bucket is None
                or record["bucket_start"] <= previous_bucket + bucket_delta
            ):
                current.append(record)
            else:
                episode_rows.append(finish_episode(domain, current))
                current = [record]
            previous_bucket = record["bucket_start"]
        if current:
            episode_rows.append(finish_episode(domain, current))

episode_columns = [
    "incident_start", "incident_end", "duration_minutes",
    "bucket_span_minutes", "domain_entity_id", "bucket_count",
    "point_count",
    "affected_entity_count", "affected_entities",
    "anomalous_metric_count", "anomalous_metrics", "max_score",
    "mean_score", "max_calibrated_score", "mean_calibrated_score",
    "max_priority_weight", "measured_fraction",
    "eligibility_reason", "eligible", "rank_score",
]
candidate_episodes = pd.DataFrame(episode_rows, columns=episode_columns)

### 5.2 Apply the evidence rule and workload budget

All candidates are saved. Only eligible candidates receive a daily rank, so rejected noise cannot consume the budget.

In [ ]:
if len(candidate_episodes):
    candidate_episodes["incident_day"] = (
        candidate_episodes["incident_start"].dt.floor("D")
    )
    candidate_episodes["daily_rank"] = pd.Series(
        pd.NA, index=candidate_episodes.index, dtype="Int64"
    )
    eligible_indexes = candidate_episodes.index[
        candidate_episodes["eligible"]
    ]
    eligible_daily_ranks = (
        candidate_episodes.loc[eligible_indexes]
        .groupby("incident_day")["rank_score"]
        .rank(method="first", ascending=False)
        .astype("Int64")
    )
    candidate_episodes.loc[
        eligible_indexes, "daily_rank"
    ] = eligible_daily_ranks
    candidate_episodes["selected_by_budget"] = (
        candidate_episodes["eligible"]
        & (
            (MAX_INCIDENTS_PER_DAY == 0)
            | candidate_episodes["daily_rank"].le(MAX_INCIDENTS_PER_DAY)
        )
    )
else:
    candidate_episodes["incident_day"] = pd.Series(
        dtype="datetime64[ns, UTC]"
    )
    candidate_episodes["daily_rank"] = pd.Series(dtype="Int64")
    candidate_episodes["selected_by_budget"] = pd.Series(dtype="bool")

candidate_episodes.to_parquet(
    MODEL_OUTPUT / "candidate_episodes.parquet", index=False
)

incident_columns = [
    "rank", "daily_rank", "incident_id", "incident_start",
    "incident_end", "duration_minutes", "bucket_span_minutes",
    "domain_entity_id", "bucket_count", "point_count",
    "affected_entity_count",
    "affected_entities", "anomalous_metric_count",
    "anomalous_metrics", "max_score", "mean_score",
    "max_calibrated_score", "mean_calibrated_score",
    "max_priority_weight", "measured_fraction",
    "eligibility_reason", "rank_score",
]
selected = candidate_episodes.loc[
    candidate_episodes["selected_by_budget"]
].copy()
if len(selected):
    incidents = (
        selected.sort_values(
            ["rank_score", "incident_start"],
            ascending=[False, True],
            kind="stable",
        )
        .reset_index(drop=True)
    )
    incidents.insert(0, "incident_id", [
        f"{SECTOR}-INC-{index:06d}"
        for index in range(1, len(incidents) + 1)
    ])
    incidents.insert(0, "rank", np.arange(1, len(incidents) + 1))
    incidents = incidents[incident_columns]
else:
    incidents = pd.DataFrame(columns=incident_columns)

incidents.to_parquet(
    MODEL_OUTPUT / "ranked_incidents.parquet", index=False
)
incidents.to_csv(
    MODEL_OUTPUT / "ranked_incidents.csv", index=False
)

display(incidents.head(20))
print("Point alerts:", len(flagged))
print("Time buckets:", len(bucket_alerts))
print("Candidate episodes:", len(candidate_episodes))
print("Eligible before budget:", int(
    candidate_episodes["eligible"].sum()
) if len(candidate_episodes) else 0)
print("Ranked incidents after budget:", len(incidents))

## 6. Freeze model and workload diagnostics

Everything in this cell is computed without `SPEC-EVAL`. The report makes the
alert-volume trade-off explicit and records how many buckets were merged,
rejected for weak evidence, or excluded by the daily budget.

In [ ]:
evaluation_phase = scored.loc[
    scored["phase"].eq("evaluation"), ["entity_id", "event_ts"]
].drop_duplicates()
evaluation_phase["observation_day"] = (
    evaluation_phase["event_ts"].dt.floor("D")
)
observed_calendar_days = max(
    1, int(evaluation_phase["observation_day"].nunique())
)
observed_entity_days = max(
    1,
    int(
        evaluation_phase[
            ["entity_id", "observation_day"]
        ].drop_duplicates().shape[0]
    ),
)
eligible_count = (
    int(candidate_episodes["eligible"].sum())
    if len(candidate_episodes) else 0
)
workload = {
    "observed_calendar_days": observed_calendar_days,
    "observed_entity_days": observed_entity_days,
    "point_alerts_per_observed_day": (
        len(flagged) / observed_calendar_days
    ),
    "eligible_episodes_per_observed_day": (
        eligible_count / observed_calendar_days
    ),
    "ranked_incidents_per_observed_day": (
        len(incidents) / observed_calendar_days
    ),
    "ranked_incidents_per_1000_entity_days": (
        len(incidents) / observed_entity_days * 1000
    ),
}

modelling_report = {
    "contract_version": CORE_VERSION,
    "week1_core_sha256": sha256_file(
        NOTEBOOK_HOME / "week1_core.py"
    ),
    "model_version": "episode_baseline_v3",
    "canonical_profile": str(PROFILE_ROOT),
    "canonical_profile_manifest_sha256": sha256_file(
        PROFILE_ROOT / "profile_manifest.json"
    ),
    "sector": SECTOR,
    "spec_core": str(CORE),
    "spec_eval_read_during_scoring": False,
    "selected_entity_count": len(selected_entities),
    "telemetry_rows_loaded": len(telemetry),
    "history": HISTORY,
    "minimum_history": MIN_HISTORY,
    "calibration_fraction": CALIBRATION_FRACTION,
    "calibration_end_min": calibration_end_min,
    "calibration_end_max": calibration_end_max,
    "target_point_quantile": TARGET_POINT_QUANTILE,
    "minimum_raw_threshold": MIN_RAW_THRESHOLD,
    "minimum_episode_buckets": MIN_EPISODE_BUCKETS,
    "high_confidence_ratio": HIGH_CONFIDENCE_RATIO,
    "maximum_incidents_per_day": MAX_INCIDENTS_PER_DAY,
    "score_rows": len(scores_to_save),
    "flagged_point_rows": len(flagged),
    "time_bucket_alerts": len(bucket_alerts),
    "candidate_episodes": len(candidate_episodes),
    "episodes_rejected_for_weak_evidence": (
        len(candidate_episodes) - eligible_count
    ),
    "eligible_episodes_before_budget": eligible_count,
    "episodes_excluded_by_daily_budget": (
        eligible_count - len(incidents)
    ),
    "ranked_incidents_after_budget": len(incidents),
    "workload": workload,
    "primary_operator_artifact": "ranked_incidents.csv",
    "supporting_artifacts": [
        "anomaly_scores.parquet",
        "calibration_windows.parquet",
        "calibration_thresholds.parquet",
        "model_diagnostics.parquet",
        "candidate_episodes.parquet",
        "ranked_incidents.parquet",
    ],
}
write_json(MODEL_OUTPUT / "modelling_report.json", modelling_report)
display(pd.Series(workload, name="value").to_frame())

## 7. Offline incident evaluation — truth is read only now

Evaluation now measures incident episodes, not merely whether any point score
existed. It reports the trade-off before and after the daily budget:

- fault-entity interval recall;
- median lead time;
- approximate incident precision;
- condition intervals overlapped by a ranked incident.

These fixture metrics prove the evaluation machinery; synthetic precision is
not evidence of real operator value.

In [ ]:
evaluation_report = {
    "status": "not_available",
    "reason": "SPEC-EVAL is absent or intentionally unmounted",
}

def evaluate_incident_set(episode_frame, interval_frame):
    if episode_frame.empty or interval_frame.empty:
        return {
            "incidents": int(len(episode_frame)),
            "fault_entity_intervals": int(len(interval_frame)),
            "matched_fault_entity_intervals": 0,
            "fault_entity_interval_recall": (
                0.0 if len(interval_frame) else None
            ),
            "incidents_overlapping_truth": 0,
            "approximate_incident_precision": (
                0.0 if len(episode_frame) else None
            ),
            "median_lead_seconds": None,
        }

    matched_intervals = 0
    lead_seconds = []
    incident_truth_hits = set()
    incident_records = episode_frame.reset_index(drop=True)

    for interval in interval_frame.itertuples(index=False):
        entity = str(interval.affected_entity_id)
        entity_calibration_end = calibration_end_by_entity[entity]
        start = max(
            pd.to_datetime(interval.active_start_ts, utc=True),
            entity_calibration_end,
        )
        end = pd.to_datetime(interval.active_end_ts, utc=True)
        if end <= start:
            continue
        matched_indexes = []
        for index, incident in incident_records.iterrows():
            affected = set(
                str(incident["affected_entities"]).split("|")
            )
            overlaps = (
                incident["incident_start"] < end
                and incident["incident_end"] > start
            )
            if entity in affected and overlaps:
                matched_indexes.append(index)
        if matched_indexes:
            matched_intervals += 1
            incident_truth_hits.update(matched_indexes)
            impact = pd.to_datetime(
                getattr(interval, "impact_ts", pd.NaT), utc=True
            )
            if pd.notna(impact):
                first_detection = incident_records.loc[
                    matched_indexes, "incident_start"
                ].min()
                lead_seconds.append(
                    (impact - first_detection).total_seconds()
                )

    return {
        "incidents": int(len(incident_records)),
        "fault_entity_intervals": int(len(interval_frame)),
        "matched_fault_entity_intervals": matched_intervals,
        "fault_entity_interval_recall": (
            matched_intervals / len(interval_frame)
            if len(interval_frame) else None
        ),
        "incidents_overlapping_truth": len(incident_truth_hits),
        "approximate_incident_precision": (
            len(incident_truth_hits) / len(incident_records)
            if len(incident_records) else None
        ),
        "median_lead_seconds": (
            float(np.median(lead_seconds))
            if lead_seconds else None
        ),
    }

calibration_end_by_entity = (
    calibration_windows.set_index("entity_id")["calibration_end"]
    .to_dict()
)

### 7.1 Open the locked truth box

Only this final modelling cell reads `SPEC-EVAL`. Deleting or unmounting that directory changes the evaluation report, but not any scores, thresholds, candidates, or ranked incidents.

In [ ]:
if EVALUATION.is_dir():
    evaluation_report = {
        "status": "evaluated_after_outputs_were_frozen",
        "spec_eval": str(EVALUATION),
        "calibration_end_min": calibration_end_min,
        "calibration_end_max": calibration_end_max,
    }
    interval_path = (
        EVALUATION / "gt_fault_entity_intervals.parquet"
    )
    if interval_path.exists():
        intervals = pd.read_parquet(interval_path)
        intervals["active_start_ts"] = pd.to_datetime(
            intervals["active_start_ts"], utc=True
        )
        intervals["active_end_ts"] = pd.to_datetime(
            intervals["active_end_ts"], utc=True
        )
        intervals = intervals.loc[
            intervals["affected_entity_id"].astype(str).isin(
                selected_entities
            )
        ].copy()
        if len(intervals):
            interval_in_evaluation = intervals.apply(
                lambda row: row["active_end_ts"]
                > calibration_end_by_entity[
                    str(row["affected_entity_id"])
                ],
                axis=1,
            )
            intervals = intervals.loc[
                interval_in_evaluation
            ].copy()

        eligible_episodes = candidate_episodes.loc[
            candidate_episodes["eligible"]
        ].copy()
        evaluation_report["before_daily_budget"] = (
            evaluate_incident_set(eligible_episodes, intervals)
        )
        evaluation_report["after_daily_budget"] = (
            evaluate_incident_set(incidents, intervals)
        )

    condition_path = (
        EVALUATION / "gt_condition_states.parquet"
    )
    if condition_path.exists():
        conditions = pd.read_parquet(condition_path)
        conditions = conditions.loc[
            conditions["entity_id"].astype(str).isin(
                selected_entities
            )
        ].copy()
        covered = 0
        evaluated_conditions = 0
        for condition in conditions.itertuples(index=False):
            entity = str(condition.entity_id)
            start = max(
                pd.to_datetime(
                    condition.condition_start_ts, utc=True
                ),
                calibration_end_by_entity[entity],
            )
            end = pd.to_datetime(
                condition.condition_end_ts, utc=True
            )
            if end <= start:
                continue
            evaluated_conditions += 1
            incident_entity_match = incidents["affected_entities"].map(
                lambda value: entity in set(str(value).split("|"))
            )
            matches = incidents.loc[
                incident_entity_match
                & incidents["incident_start"].lt(end)
                & incidents["incident_end"].gt(start)
            ]
            covered += int(len(matches) > 0)
        evaluation_report["condition_intervals"] = (
            evaluated_conditions
        )
        evaluation_report[
            "condition_intervals_overlapped_by_ranked_incident"
        ] = covered

write_json(
    MODEL_OUTPUT / "offline_evaluation.json",
    evaluation_report,
)
display(pd.Series(evaluation_report, name="value").to_frame())

## 8. How to use the result

Start with the profile outputs. Confirm that missingness, clipping, degenerate series, and transform recommendations are credible. Seasonality evidence is descriptive; this baseline does not remove seasonality automatically.

Then start with `modelling_report.json`:

- if eligible episodes per day remain excessive, improve calibration and
  episode rules before interpreting ranks;
- if the daily budget materially reduces recall, the model needs better
  evidence—not merely a larger budget;
- inspect `model_diagnostics.parquet` for one metric dominating alerts;
- inspect `candidate_episodes.parquet` to see what the evidence filter or
  budget removed.

Then review the top 20–50 rows of `ranked_incidents.csv`. Mark each as
technically coherent, likely artefact, merge, suppress, or missing context.

**Next model challengers:** seasonal baselines for
`multiple_seasonalities`, count distributions for
`overdispersed_count`, a proper hurdle model for
`zero_inflated_bounded`, and finally a multivariate challenger. Each must beat
this frozen episode baseline at incident-level recall and alert workload.